# 생성형 인공지능 — 11주차 실습
## 사전 학습 언어 모델 (Pre-trained Language Models)

**대상**: 학부 4학년 | **환경**: 로컬 Jupyter/VS Code | **소요 시간**: 약 90~120분

---

### 실습 목표

| # | 문제 | 강의 대응 내용 |
|---|------|----------------|
| 1 | 사전 학습 언어 모델 지도 그리기 | 인코더/디코더/인코더-디코더, 모델별 핵심 아이디어 |
| 2 | 토크나이저와 BERT의 입력 구조 | BERT의 구조, 두 개의 문장 분류 과업 |
| 3 | MLM(마스크드 언어 모델) 직접 구현 | BERT의 80/10/10 규칙, RoBERTa의 동적 마스킹 |
| 4 | 인코더 vs 디코더 (BERT vs GPT-2) | 양방향 인코더와 단방향(마스크드) 디코더 |
| 5 | 모델 경량화 (ALBERT / DistilBERT) | 임베딩 행렬 분해, 모수 공유, 지식 증류 |
| 6 | ELECTRA의 교체된 토큰 탐지(RTD) | 생성기-판별기 구조 |
| 7 | T5의 Text-to-Text 통합 | 제어(접두) 코드, 다양한 길이의 빈칸 채우기 |
| 8 | DistilBERT 미세조정 (감성 분석) | 사전 학습 → 하위 과업 미세조정 |

### 진행 방법

- `# TODO` 로 표시된 부분을 채워 넣으세요.
- 각 문제 끝에는 **자가 채점 셀**이 있습니다. `모든 검증 통과` 가 출력되면 정답입니다.
- 문제 8은 GPU가 필요합니다. VS Code에서 `rl2026` 커널을 선택하세요.
  (문제 1~7은 CPU에서도 동작합니다.)

> **[문제 노트북]** `# TODO` 부분을 채워 넣으세요.

---
## 0. 환경 설정

아래 셀을 먼저 실행하세요. `rl2026` 커널과 인터넷 연결이 필요합니다.

In [41]:
import importlib.util

required = ["transformers", "datasets", "sentencepiece", "torch"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(
        "다음 패키지가 현재 커널에 없습니다: " + ", ".join(missing) +
        ". rl2026 커널에 설치한 뒤 다시 실행하세요."
    )
print("로컬 패키지 확인 완료:", ", ".join(required))

로컬 패키지 확인 완료: transformers, datasets, sentencepiece, torch


In [42]:
import os, time, math, random
import numpy as np
import torch
import torch.nn.functional as F
import transformers

os.environ["TOKENIZERS_PARALLELISM"] = "false"
transformers.logging.set_verbosity_error()   # 경고 메시지 최소화

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers :", transformers.__version__)
print("torch        :", torch.__version__)
print("device       :", DEVICE)

transformers : 5.22.2
torch        : 2.11.0+cu128
device       : cuda


---
# 문제 1. 사전 학습 언어 모델 지도 그리기 (개념 정리)

강의에서 다룬 모델들은 크게 **트랜스포머의 어느 부분을 사용하는가**로 나뉩니다.

- **인코더 구조**: 빈칸을 예측하기 위해 앞/뒤 문맥을 모두 참조 → 문장 분류, 감성 분석, 질의응답에 강함
- **디코더 구조**: 주어진 문장 다음에 오는 단어를 예측 → 생성 과업에 특화
- **인코더-디코더 구조**: 트랜스포머 원형과 동일 → 번역, 요약, 챗봇 등 seq2seq 과업에 우수

### 1-1. 각 모델을 구조별로 분류하세요.

`ARCH` 딕셔너리의 값을 `"encoder"`, `"decoder"`, `"encoder-decoder"` 중 하나로 채우세요.

In [43]:
ARCH = {
    "BERT":         "encoder",
    "RoBERTa":      "encoder",
    "ALBERT":       "encoder",
    "DistilBERT":   "encoder",
    "ELECTRA":      "encoder",
    "DeBERTa":      "encoder",
    "SpanBERT":     "encoder",
    "MobileBERT":   "encoder",
    "GPT-2":        "decoder",
    "CTRL":         "decoder",
    "TransformerXL": "decoder",
    "BART":         "encoder-decoder",
    "T5":           "encoder-decoder",
}

### 1-2. 각 모델의 '핵심 아이디어'를 연결하세요.

`IDEA` 딕셔너리에서 모델 이름을 key로, 아래 보기의 **번호**를 value로 채우세요.

```
1. 동적 마스킹 (매 학습 시 마스크 위치가 바뀜)
2. 임베딩 행렬 분해 + 인코더 층 간 모수 공유
3. 지식 증류 (선생님 모델 → 학생 모델)
4. 연속된 구간(Span) 마스킹 + Span 경계 목적
5. 교체된 토큰 탐지 (생성기가 만든 토큰을 판별기가 구분)
6. 분리된 어텐션 (토큰의 의미와 상대 위치 정보를 분리)
7. 세그먼트 레벨 순환 (긴 입력을 세그먼트로 나눠 재귀적으로 연산)
8. 순열 언어 모델 + Two-Stream Self-Attention
9. 제어 코드를 앞에 붙여 조건부 생성을 제어
10. 모든 자연어 처리 과업을 텍스트-투-텍스트로 통일
11. 입력에 노이즈를 준 뒤 원본으로 복원하며 학습
12. 병목(bottleneck) 구조로 인코더 내부 모수를 대폭 축소한 경량 모델
```

In [44]:
IDEA = {
    "RoBERTa":       1,
    "ALBERT":        2,
    "DistilBERT":    3,
    "SpanBERT":      4,
    "ELECTRA":       5,
    "DeBERTa":       6,
    "TransformerXL": 7,
    "XLNet":         8,
    "CTRL":          9,
    "T5":            10,
    "BART":          11,
    "MobileBERT":    12,
}

In [45]:
# ===== 자가 채점 (문제 1) =====
_arch = {"BERT":"encoder","RoBERTa":"encoder","ALBERT":"encoder","DistilBERT":"encoder",
         "ELECTRA":"encoder","DeBERTa":"encoder","SpanBERT":"encoder","MobileBERT":"encoder",
         "GPT-2":"decoder","CTRL":"decoder","TransformerXL":"decoder",
         "BART":"encoder-decoder","T5":"encoder-decoder"}
_idea = {"RoBERTa":1,"ALBERT":2,"DistilBERT":3,"SpanBERT":4,"ELECTRA":5,"DeBERTa":6,
         "TransformerXL":7,"XLNet":8,"CTRL":9,"T5":10,"BART":11,"MobileBERT":12}

wrong = [k for k, v in _arch.items() if ARCH.get(k) != v]
assert not wrong, f"1-1 오답: {wrong}"
wrong = [k for k, v in _idea.items() if IDEA.get(k) != v]
assert not wrong, f"1-2 오답: {wrong}"
print("문제 1: 모든 검증 통과")

문제 1: 모든 검증 통과


---
# 문제 2. 토크나이저와 BERT의 입력 구조

BERT는 다양한 자연어 처리 과업을 수행하기 위해 **최대 두 개의 연속된 문장**을 입력으로 받습니다.
이때 입력은 다음과 같이 구성됩니다.

```
[CLS] 문장A 토큰들 [SEP] 문장B 토큰들 [SEP]
token_type_ids:  0 0 ... 0   0     1 1 ... 1   1
```

- `[CLS]`: 문장 전체를 대표하는 토큰. 분류 과업에서 이 위치의 출력 벡터를 사용합니다.
- `[SEP]`: 문장의 경계를 표시합니다.
- `token_type_ids`(세그먼트 임베딩): 첫 번째 문장은 0, 두 번째 문장은 1.

### 2-1. 두 문장을 BERT 입력 형식으로 인코딩하는 함수를 완성하세요.

In [46]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

def encode_pair(sent_a, sent_b):
    '''두 문장을 BERT 입력 형식으로 인코딩한다.
    반환: (tokens, input_ids, token_type_ids)  -- 모두 파이썬 리스트
    '''
    #       힌트: tok(sent_a, sent_b) 는 input_ids / token_type_ids / attention_mask 를 반환합니다.
    enc = tok(sent_a, sent_b, add_special_tokens=True)
    ids = enc["input_ids"]
    ttypes = enc["token_type_ids"]
    tokens = tok.convert_ids_to_tokens(ids)
    return tokens, ids, ttypes


A = "a man is riding a horse."
B = "a person is riding a bicycle."
tokens, pair_ids, ttypes = encode_pair(A, B)

for t, i, s in zip(tokens, pair_ids, ttypes):
    print(f"{t:>12s}  id={i:<6d} segment={s}")

       [CLS]  id=101    segment=0
           a  id=1037   segment=0
         man  id=2158   segment=0
          is  id=2003   segment=0
      riding  id=5559   segment=0
           a  id=1037   segment=0
       horse  id=3586   segment=0
           .  id=1012   segment=0
       [SEP]  id=102    segment=0
           a  id=1037   segment=1
      person  id=2711   segment=1
          is  id=2003   segment=1
      riding  id=5559   segment=1
           a  id=1037   segment=1
     bicycle  id=10165  segment=1
           .  id=1012   segment=1
       [SEP]  id=102    segment=1


### 2-2. BERT-BASE의 임베딩 행렬 크기를 계산하세요.

강의자료에서 BERT의 임베딩 행렬 크기는 `토큰의 수 30,522 × 임베딩 벡터의 차원 768 = 23,440,896` 이었습니다.
실제 모델에서 이 값을 확인하고, 전체 파라미터 수 중 임베딩이 차지하는 비율을 구하세요.

In [47]:
from transformers import AutoModel

bert = AutoModel.from_pretrained("bert-base-uncased")

V = bert.config.vocab_size      # 토큰의 수
H = bert.config.hidden_size     # 임베딩 벡터(은닉 상태)의 차원
L = bert.config.num_hidden_layers

word_emb_params = V * H

#       힌트: bert.parameters() 를 순회하며 p.numel() 을 더합니다.
total_params = sum(p.numel() for p in bert.parameters())

print(f"vocab_size={V}, hidden_size={H}, num_layers={L}")
print(f"워드 임베딩 파라미터 : {word_emb_params:,}")
print(f"전체 파라미터        : {total_params:,}")
print(f"임베딩 비중          : {word_emb_params / total_params * 100:.1f}%")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1488.22it/s]

vocab_size=30522, hidden_size=768, num_layers=12
워드 임베딩 파라미터 : 23,440,896
전체 파라미터        : 109,482,240
임베딩 비중          : 21.4%


In [48]:
# ===== 자가 채점 (문제 2) =====
assert tokens[0] == "[CLS]", "첫 토큰은 [CLS] 여야 합니다."
assert tokens[-1] == "[SEP]", "마지막 토큰은 [SEP] 여야 합니다."
assert tokens.count("[SEP]") == 2, "문장 쌍 입력에는 [SEP]이 2개 있어야 합니다."
assert pair_ids[0] == 101 and pair_ids[-1] == 102, "[CLS]=101, [SEP]=102 이어야 합니다."
assert set(ttypes) == {0, 1}, "token_type_ids 는 0과 1로 구성되어야 합니다."
# 첫 번째 [SEP]까지가 segment 0
sep1 = tokens.index("[SEP]")
assert all(s == 0 for s in ttypes[:sep1 + 1]) and all(s == 1 for s in ttypes[sep1 + 1:]), \
    "세그먼트 구분이 올바르지 않습니다."
assert word_emb_params == 30522 * 768 == 23_440_896, "임베딩 행렬 크기가 강의자료와 다릅니다."
assert 1.08e8 < total_params < 1.11e8, "BERT-BASE 전체 파라미터는 약 1.1억 개입니다."
print("문제 2: 모든 검증 통과")

문제 2: 모든 검증 통과


---
# 문제 3. 마스크드 언어 모델(MLM) 직접 구현하기

BERT는 **전체 토큰의 15%** 를 무작위로 골라 예측 대상으로 삼습니다.
그런데 선택된 토큰을 항상 `[MASK]`로 바꾸면, 미세조정 시에는 `[MASK]`가 등장하지 않아
사전 학습과 미세조정 사이에 불일치가 생깁니다. 이를 보완하기 위해 다음 규칙을 사용합니다.

| 비율 | 처리 |
|------|------|
| 80% | 그대로 `[MASK]` 토큰으로 변경 |
| 10% | 무작위 토큰으로 변경 |
| 10% | 원본 그대로 유지 |

### 3-1. 위 80/10/10 규칙을 구현하세요.

In [49]:
def mlm_mask(input_ids, tokenizer, mlm_prob=0.15, rng=None):
    '''BERT 방식의 마스킹을 수행한다.

    input_ids : 1차원 LongTensor  (특수 토큰 포함)
    반환      : (masked_ids, labels)
                labels 는 예측 대상이 아닌 위치를 -100 으로 채운다.
                (-100 은 PyTorch CrossEntropyLoss 의 기본 ignore_index)
    '''
    if rng is None:
        rng = torch.Generator().manual_seed(0)

    labels = input_ids.clone()
    masked = input_ids.clone()

    # (1) 각 토큰을 15% 확률로 예측 대상으로 선택
    prob = torch.full(labels.shape, mlm_prob)
    # 특수 토큰([CLS], [SEP] 등)은 예측 대상에서 제외
    special = torch.tensor(
        tokenizer.get_special_tokens_mask(input_ids.tolist(), already_has_special_tokens=True),
        dtype=torch.bool)
    prob.masked_fill_(special, 0.0)
    selected = torch.bernoulli(prob, generator=rng).bool()

    # 예측 대상이 아닌 위치는 손실 계산에서 제외
    labels[~selected] = -100

    # (2) 선택된 토큰 중 80% -> [MASK]
    replace_mask = torch.bernoulli(torch.full(labels.shape, 0.8), generator=rng).bool() & selected
    masked[replace_mask] = tokenizer.mask_token_id

    # (3) 남은 20% 중 절반(= 전체의 10%) -> 무작위 토큰
    replace_random = torch.bernoulli(torch.full(labels.shape, 0.5), generator=rng).bool() & selected & ~replace_mask
    random_tokens = torch.randint(len(tokenizer), labels.shape, dtype=torch.long, generator=rng)
    masked[replace_random] = random_tokens[replace_random]

    # (4) 나머지 10% 는 원본 그대로 둔다 (아무 것도 하지 않음)
    return masked, labels


# 동작 확인
sent = "i was hungry so i ate an apple after the long meeting yesterday."
mlm_ids = tok(sent, return_tensors="pt")["input_ids"][0]
g = torch.Generator().manual_seed(7)
m_ids, lbls = mlm_mask(mlm_ids, tok, rng=g)

print("원본  :", " ".join(tok.convert_ids_to_tokens(mlm_ids)))
print("마스킹:", " ".join(tok.convert_ids_to_tokens(m_ids)))
print("예측 대상 위치:", (lbls != -100).nonzero().flatten().tolist())

원본  : [CLS] i was hungry so i ate an apple after the long meeting yesterday . [SEP]
마스킹: [CLS] i was hungry so i ate an apple after the long meeting yesterday . [SEP]
예측 대상 위치: []


### 3-2. 실제 BERT로 빈칸을 채워 보세요.

강의자료의 예시 `"나는 배가 고팠다. 그래서 ___ 먹었다."` 처럼,
`[MASK]` 위치에 대한 상위 5개 예측 단어와 확률을 출력하세요.

In [50]:
from transformers import AutoModelForMaskedLM

mlm_model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased").eval()

def predict_mask(text, topk=5):
    '''text 안의 [MASK] 위치에 대한 상위 topk 예측을 (토큰, 확률) 리스트로 반환'''
    enc = tok(text, return_tensors="pt")
    with torch.no_grad():
        logits = mlm_model(**enc).logits          # (1, seq_len, vocab_size)

    mask_pos = (enc["input_ids"] == tok.mask_token_id).nonzero(as_tuple=True)[1]
    probs = torch.softmax(logits[0, mask_pos.item()], dim=-1)
    top = torch.topk(probs, k=topk)
    return [(tok.convert_ids_to_tokens(int(i)), float(p)) for p, i in zip(top.values, top.indices)]


for s in ["i was hungry so i ate an [MASK].",
          "paris is the capital of [MASK].",
          "the movie was absolutely [MASK]."]:
    print(s)
    for w, p in predict_mask(s):
        print(f"    {w:<12s} {p:.3f}")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1534.03it/s]


i was hungry so i ate an [MASK].
    apple        0.471
    hour         0.095
    orange       0.094
    egg          0.090
    olive        0.029
paris is the capital of [MASK].
    france       0.951
    algeria      0.012
    morocco      0.003
    reunion      0.003
    haiti        0.003
the movie was absolutely [MASK].
    fantastic    0.093
    perfect      0.060
    beautiful    0.058
    amazing      0.053
    fabulous     0.052


### 3-3. 정적 마스킹 vs 동적 마스킹 (RoBERTa)

BERT는 전처리 단계에서 마스크 위치를 한 번 정해 두고 학습 내내 재사용합니다(**정적 마스킹**).
RoBERTa는 같은 문장이라도 학습 스텝마다 마스크 위치를 새로 뽑습니다(**동적 마스킹**).

아래 코드를 완성해, 같은 문장에 대해 두 방식이 만들어 내는 마스크 패턴이 어떻게 다른지 확인하세요.

In [51]:
def static_masking(ids, n_epochs=4):
    '''BERT 방식: 마스크를 한 번만 만들고 매 에폭 재사용'''
    g = torch.Generator().manual_seed(123)
    m, _ = mlm_mask(ids, tok, rng=g)
    return [m.clone() for _ in range(n_epochs)]

def dynamic_masking(ids, n_epochs=4):
    '''RoBERTa 방식: 매 에폭마다 마스크를 새로 생성'''
    out = []
    for epoch in range(n_epochs):
        g = torch.Generator().manual_seed(123 + epoch)
        m, _ = mlm_mask(ids, tok, rng=g)
        out.append(m.clone())
    return out


def masked_slots(m):
    return tuple((m == tok.mask_token_id).nonzero().flatten().tolist())

s_patterns = [masked_slots(m) for m in static_masking(mlm_ids)]
d_patterns = [masked_slots(m) for m in dynamic_masking(mlm_ids)]

print("정적 마스킹 (BERT)")
for i, p in enumerate(s_patterns):
    print(f"  epoch {i}: {p}")
print("서로 다른 패턴 수:", len(set(s_patterns)))

print("\n동적 마스킹 (RoBERTa)")
for i, p in enumerate(d_patterns):
    print(f"  epoch {i}: {p}")
print("서로 다른 패턴 수:", len(set(d_patterns)))

정적 마스킹 (BERT)
  epoch 0: (6,)
  epoch 1: (6,)
  epoch 2: (6,)
  epoch 3: (6,)
서로 다른 패턴 수: 1

동적 마스킹 (RoBERTa)
  epoch 0: (6,)
  epoch 1: (5, 10)
  epoch 2: (3,)
  epoch 3: (10,)
서로 다른 패턴 수: 4


In [52]:
# ===== 자가 채점 (문제 3) =====
# (a) 대규모 표본(5회 평균)으로 80/10/10 비율 확인
long_ids = tok(" ".join(["the quick brown fox jumps over the lazy dog"] * 60),
               return_tensors="pt", truncation=True, max_length=512)["input_ids"][0]

tot_sel = tot_mask = tot_keep = 0
for trial in range(5):
    mm, ll = mlm_mask(long_ids, tok, rng=torch.Generator().manual_seed(2024 + trial))
    sel = (ll != -100)
    tot_sel  += int(sel.sum())
    tot_mask += int(((mm == tok.mask_token_id) & sel).sum())
    tot_keep += int(((mm == long_ids) & sel).sum())
    assert int(((mm != long_ids) & ~sel).sum()) == 0, "예측 대상이 아닌 토큰을 바꾸면 안 됩니다."
tot_rand = tot_sel - tot_mask - tot_keep

assert tot_sel > 150, "선택된 토큰이 너무 적습니다. 15% 선택이 동작하는지 확인하세요."
print(f"선택 {tot_sel}개 -> [MASK] {tot_mask/tot_sel:.0%}, "
      f"무작위 {tot_rand/tot_sel:.0%}, 유지 {tot_keep/tot_sel:.0%}")
assert 0.62 < tot_mask / tot_sel < 0.95, "[MASK] 비율이 80%에서 크게 벗어납니다."
assert 0.01 < tot_rand / tot_sel < 0.25, "무작위 교체 비율이 10%에서 크게 벗어납니다."

# (b) fill-mask 상식 확인 (상위 5개 안에 정답이 있으면 통과)
top5 = [w for w, _ in predict_mask("paris is the capital of [MASK].")]
assert "france" in top5, f"예상 밖의 예측: {top5}"
_top5 = predict_mask("the movie was absolutely [MASK].", topk=5)
_ps = [p for _, p in _top5]
assert all(0.0 < p < 1.0 for p in _ps), "확률 값이 0~1 범위를 벗어납니다. softmax 를 적용했는지 확인하세요."
assert _ps == sorted(_ps, reverse=True), "topk 결과는 확률 내림차순이어야 합니다."

# (c) 정적/동적 마스킹
assert len(set(s_patterns)) == 1, "정적 마스킹은 매 에폭 동일한 패턴이어야 합니다."
assert len(set(d_patterns)) > 1, "동적 마스킹은 에폭마다 패턴이 달라야 합니다."
print("문제 3: 모든 검증 통과")

선택 358개 -> [MASK] 80%, 무작위 11%, 유지 9%
문제 3: 모든 검증 통과


---
# 문제 4. 인코더 vs 디코더 — BERT와 GPT-2는 무엇이 다른가

강의자료의 두 그림을 코드로 확인합니다.

- **인코더 구조(BERT)**: 빈칸을 예측하기 위해 **앞/뒤 문맥을 모두** 참조
- **디코더 구조(GPT)**: 주어진 문장 **다음에 오는 단어**를 예측 → 미래 토큰을 볼 수 없도록
  **마스크드 멀티헤드 어텐션**을 사용

### 4-1. 디코더의 마스크드 어텐션 마스크를 직접 만들어 보세요.

길이 `n`의 시퀀스에서 `i`번째 토큰은 `0..i` 까지만 볼 수 있어야 합니다.

In [53]:
def causal_mask(n):
    '''(n, n) bool 텐서. True = 참조 가능한 위치'''
    #       힌트: torch.tril, torch.ones
    return torch.tril(torch.ones(n, n, dtype=torch.bool))

def bidirectional_mask(n):
    '''인코더(BERT)의 어텐션 마스크: 모든 위치를 참조 가능'''
    return torch.ones(n, n, dtype=torch.bool)

n = 6
print("[디코더 - GPT] 마스크드 어텐션")
print(causal_mask(n).int().numpy())
print("\n[인코더 - BERT] 양방향 어텐션")
print(bidirectional_mask(n).int().numpy())
print("\n참조 가능한 (질의, 키) 쌍의 수 :",
      int(causal_mask(n).sum()), "vs", int(bidirectional_mask(n).sum()))

[디코더 - GPT] 마스크드 어텐션
[[1 0 0 0 0 0]
 [1 1 0 0 0 0]
 [1 1 1 0 0 0]
 [1 1 1 1 0 0]
 [1 1 1 1 1 0]
 [1 1 1 1 1 1]]

[인코더 - BERT] 양방향 어텐션
[[1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]]

참조 가능한 (질의, 키) 쌍의 수 : 21 vs 36


### 4-2. GPT-2로 '다음 단어'를 예측하세요.

디코더 구조는 본질적인 언어 모델과 유사하게 **다음에 올 단어**를 예측합니다.

In [54]:
from transformers import AutoModelForCausalLM

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def next_token_topk(prefix, topk=5):
    '''prefix 다음에 올 토큰 상위 topk 를 (토큰, 확률)로 반환'''
    enc = gpt2_tok(prefix, return_tensors="pt")
    with torch.no_grad():
        logits = gpt2(**enc).logits            # (1, seq_len, vocab)

    #       (디코더는 i번째 위치의 출력이 i+1번째 토큰의 분포를 의미합니다.)
    probs = torch.softmax(logits[0, -1], dim=-1)
    top = torch.topk(probs, k=topk)
    return [(gpt2_tok.decode([int(i)]), float(p)) for p, i in zip(top.values, top.indices)]


for prefix in ["I was hungry, so I ate an", "The capital of France is"]:
    print(f"[{prefix} ___]")
    for w, p in next_token_topk(prefix):
        print(f"    {w!r:<14s} {p:.3f}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 893.89it/s]


[I was hungry, so I ate an ___]
    ' apple'       0.143
    ' egg'         0.134
    ' entire'      0.083
    ' orange'      0.055
    ' extra'       0.047
[The capital of France is ___]
    ' the'         0.085
    ' now'         0.048
    ' a'           0.046
    ' France'      0.032
    ' Paris'       0.032


### 4-3. 양방향성이 왜 중요한가?

아래 두 문장은 **빈칸의 왼쪽 문맥이 완전히 동일**하고 오른쪽만 다릅니다.

```
he opened the [MASK] and started to read .
he opened the [MASK] and drove away .
```

- BERT(인코더)는 오른쪽 문맥까지 보므로 서로 **다른** 예측을 내놓아야 합니다.
- GPT-2(디코더)는 `"he opened the"` 까지만 보므로 두 경우의 예측이 **완전히 같습니다**.

이를 코드로 확인하세요.

In [55]:
s1 = "he opened the [MASK] and started to read ."
s2 = "he opened the [MASK] and drove away ."

try:
    bert_top5_s1 = [w for w, _ in predict_mask(s1)]
    bert_top5_s2 = [w for w, _ in predict_mask(s2)]
    print(f"BERT  : {s1}\n        -> {bert_top5_s1}")
    print(f"BERT  : {s2}\n        -> {bert_top5_s2}")
except Exception as e:
    print("문제 3-2 의 predict_mask 를 먼저 완성하세요:", e)

#       두 문장에서 GPT-2 가 실제로 참조할 수 있는 공통 접두사를 문자열로 지정하세요.
gpt2_prefix = "he opened the"

gpt2_top1 = next_token_topk(gpt2_prefix)[0][0]
print(f"\nGPT-2 : '{gpt2_prefix}' -> {gpt2_top1!r}  (두 문장 모두 동일한 예측)")

BERT  : he opened the [MASK] and started to read .
        -> ['book', 'journal', 'paper', 'letter', 'magazine']
BERT  : he opened the [MASK] and drove away .
        -> ['door', 'trunk', 'gate', 'doors', 'window']

GPT-2 : 'he opened the' -> ' door'  (두 문장 모두 동일한 예측)


In [56]:
# ===== 자가 채점 (문제 4) =====
cm, bm = causal_mask(5), bidirectional_mask(5)
assert cm.dtype == torch.bool and bm.dtype == torch.bool, "마스크는 bool 텐서여야 합니다."
assert torch.equal(cm, torch.tril(torch.ones(5, 5, dtype=torch.bool))), "하삼각 마스크가 아닙니다."
assert int(cm.sum()) == 15 and int(bm.sum()) == 25, "참조 가능한 쌍의 수가 맞지 않습니다."
assert bool(cm[0, 1]) is False and bool(cm[1, 0]) is True, "미래 토큰을 참조하고 있습니다."

g2 = next_token_topk("The capital of France is")
assert len(g2) == 5 and all(0.0 < p < 1.0 for _, p in g2), "확률 값이 올바르지 않습니다."
assert [p for _, p in g2] == sorted([p for _, p in g2], reverse=True), "확률 내림차순이 아닙니다."
print("참고 - GPT-2 top5:", [w for w, _ in g2], "  (소형 모델이라 'Paris'가 없을 수도 있습니다)")

assert bert_top5_s1 != bert_top5_s2, \
    "BERT는 오른쪽 문맥이 다르면 다른 예측을 내야 합니다 (양방향성)."
assert gpt2_prefix.strip().lower() == "he opened the", \
    "GPT-2가 볼 수 있는 것은 빈칸 왼쪽의 공통 접두사뿐입니다."
print("BERT top-1 :", bert_top5_s1[0], "/", bert_top5_s2[0], " -> 오른쪽 문맥에 따라 달라짐")
print("GPT-2      :", gpt2_top1, " -> 두 문장에서 동일 (왼쪽 문맥만 참조)")
print("문제 4: 모든 검증 통과")

참고 - GPT-2 top5: [' the', ' now', ' a', ' France', ' Paris']   (소형 모델이라 'Paris'가 없을 수도 있습니다)
BERT top-1 : book / door  -> 오른쪽 문맥에 따라 달라짐
GPT-2      :  door  -> 두 문장에서 동일 (왼쪽 문맥만 참조)
문제 4: 모든 검증 통과


---
# 문제 5. 모델 경량화 — ALBERT와 DistilBERT

### 5-1. ALBERT의 임베딩 행렬 분해

BERT는 어휘 사전 크기 `V`, 은닉 차원 `H`에 대해 `V x H` 크기의 임베딩 행렬을 가집니다.
ALBERT는 이를 더 작은 임베딩 차원 `E`를 거치도록 **두 개의 행렬로 분해**합니다.

```
BERT   : V x H
ALBERT : (V x E) + (E x H)      (E << H)
```

강의자료 기준: `V = 30,522`, `H = 768`, `E = 64` 또는 `128`.

In [57]:
V, H = 30522, 768

def bert_embedding_params(V, H):
    return V * H

def albert_embedding_params(V, H, E):
    return V * E + E * H

print(f"BERT   (V x H)          : {bert_embedding_params(V, H):,}")
for E in (64, 128):
    p = albert_embedding_params(V, H, E)
    print(f"ALBERT (E={E:3d})          : {p:,}   ({bert_embedding_params(V, H)/p:.1f}배 감소)")

BERT   (V x H)          : 23,440,896
ALBERT (E= 64)          : 2,002,560   (11.7배 감소)
ALBERT (E=128)          : 4,005,120   (5.9배 감소)


### 5-2. 실제 모델로 확인하기

ALBERT는 임베딩 분해에 더해 **인코더 층 간 모수 공유**(하나의 인코더를 재귀적으로 반복)를 적용합니다.
BERT-BASE는 12개의 서로 다른 인코더를 쌓지만, ALBERT는 12번 반복하더라도 파라미터는 1개 층 분량만 갖습니다.

In [58]:
from transformers import AutoModel

def load_info(name):
    m = AutoModel.from_pretrained(name)
    cfg = m.config
    n = sum(p.numel() for p in m.parameters())
    return m, cfg, n

# 문제 2에서 이미 불러온 bert 를 재사용한다 (메모리 절약)
bert_m,  bert_cfg,  bert_n  = bert, bert.config, sum(p.numel() for p in bert.parameters())
alb_m,   alb_cfg,   alb_n   = load_info("albert-base-v2")
dist_m,  dist_cfg,  dist_n  = load_info("distilbert-base-uncased")

rows = [
    ("BERT-BASE",  bert_cfg.num_hidden_layers, bert_n),
    ("ALBERT-BASE", alb_cfg.num_hidden_layers,  alb_n),
    ("DistilBERT",  dist_cfg.num_hidden_layers, dist_n),
]
print(f"{'모델':<12}{'인코더 층':>10}{'파라미터':>16}{'BERT 대비':>12}")
for name, L_, n_ in rows:
    print(f"{name:<12}{L_:>10}{n_:>16,}{n_/bert_n*100:>11.1f}%")

albert_E = alb_cfg.embedding_size
print(f"\nALBERT embedding_size E = {albert_E}, hidden_size H = {alb_cfg.hidden_size}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2073.45it/s]

모델               인코더 층            파라미터     BERT 대비
BERT-BASE           12     109,482,240      100.0%
ALBERT-BASE         12      11,683,584       10.7%
DistilBERT           6      66,362,880       60.6%

ALBERT embedding_size E = 128, hidden_size H = 768


### 5-3. DistilBERT — 지식 증류의 효과

DistilBERT는 BERT-BASE(선생님 모델)의 출력 분포를 학생 모델이 학습하는 **지식 증류** 방식으로 만들어졌습니다.
인코더 층 수가 12 → 6으로 줄었을 때 **추론 속도**가 얼마나 빨라지는지 측정하세요.

In [59]:
import time

def benchmark(model, n_iter=20, seq_len=128, batch=8):
    '''forward 1회당 평균 소요 시간(ms)을 반환'''
    model = model.eval()
    x = torch.randint(1000, 2000, (batch, seq_len))
    with torch.no_grad():
        model(input_ids=x)                      # 워밍업
    #       힌트: time.perf_counter()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iter):
            model(input_ids=x)
    return (time.perf_counter() - start) * 1000 / n_iter


t_bert = benchmark(bert_m)
t_dist = benchmark(dist_m)
print(f"BERT-BASE  : {t_bert:7.1f} ms / forward")
print(f"DistilBERT : {t_dist:7.1f} ms / forward   ->  {t_bert/t_dist:.2f}배 빠름")
print(f"파라미터    : {dist_n/bert_n*100:.1f}% (약 {100 - dist_n/bert_n*100:.0f}% 감소)")

BERT-BASE  :   397.4 ms / forward
DistilBERT :   189.5 ms / forward   ->  2.10배 빠름
파라미터    : 60.6% (약 39% 감소)


In [60]:
# ===== 자가 채점 (문제 5) =====
assert bert_embedding_params(30522, 768) == 23_440_896, "강의자료의 23,440,896 과 일치해야 합니다."
assert albert_embedding_params(30522, 768, 128) == 30522 * 128 + 128 * 768
assert albert_embedding_params(30522, 768, 64) < albert_embedding_params(30522, 768, 128)
assert bert_embedding_params(V, H) / albert_embedding_params(V, H, 128) > 5, \
    "ALBERT 분해로 임베딩 파라미터가 5배 이상 줄어야 합니다."

assert albert_E == 128, "ALBERT-BASE 의 embedding_size 는 128 입니다."
assert alb_cfg.num_hidden_layers == 12 and alb_n < bert_n * 0.2, \
    "ALBERT는 12층이지만 모수 공유로 파라미터가 크게 적습니다."
assert dist_cfg.num_hidden_layers == 6, "DistilBERT 는 6개의 인코더를 갖습니다."
assert 0.5 < dist_n / bert_n < 0.7, "DistilBERT 파라미터는 BERT의 약 60% 수준입니다."
assert float(t_bert) > 0 and float(t_dist) > 0, "벤치마크 결과가 올바르지 않습니다."
assert t_dist < t_bert, "DistilBERT 가 더 빨라야 합니다."
print("문제 5: 모든 검증 통과")

문제 5: 모든 검증 통과


---
# 문제 6. ELECTRA — 교체된 토큰 탐지 (Replaced Token Detection)

ELECTRA는 GAN에서 착안한 구조를 사용합니다.

```
원본 문장 -> 일부를 [MASK] -> [생성기(소형 MLM)] -> 그럴듯한 토큰으로 채움
                                       |
                                       v
                         [판별기(ELECTRA)] -> 각 토큰이 '원본'인가 '교체됨'인가?
```

중요한 점: **생성기가 채운 토큰이 원본과 같으면 그 토큰의 정답 라벨은 '원본'** 입니다
(강의자료: "생성기에서 [MASK] 토큰을 예측한 토큰이 입력과 동일하면 교체된 토큰으로 예측하지 않음").

판별기는 **모든 토큰**에 대해 학습 신호를 받기 때문에, 15%만 사용하는 MLM보다 학습 효율이 높습니다.

### 6-1. 생성기로 손상된(corrupted) 문장 만들기

In [61]:
from transformers import ElectraTokenizerFast, ElectraForMaskedLM, ElectraForPreTraining

e_tok = ElectraTokenizerFast.from_pretrained("google/electra-small-generator")
generator = ElectraForMaskedLM.from_pretrained("google/electra-small-generator").eval()
discriminator = ElectraForPreTraining.from_pretrained("google/electra-small-discriminator").eval()

sentence = ("the chef cooked the meal in the small kitchen and the guests "
            "enjoyed every bite of the delicious dinner")
orig_ids = e_tok(sentence, return_tensors="pt")["input_ids"]      # (1, L)

# 마스킹할 위치를 직접 지정 (특수 토큰 제외)
MASK_POS = [2, 3, 9, 13, 18]
masked_ids = orig_ids.clone()
masked_ids[0, MASK_POS] = e_tok.mask_token_id

with torch.no_grad():
    gen_logits = generator(input_ids=masked_ids).logits            # (1, L, V)

corrupted_ids = masked_ids.clone()
for pos in MASK_POS:
    corrupted_ids[0, pos] = gen_logits[0, pos].argmax()

print("원본      :", e_tok.decode(orig_ids[0]))
print("마스킹    :", e_tok.decode(masked_ids[0]))
print("생성기 결과:", e_tok.decode(corrupted_ids[0]))

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 47046.29it/s]

원본      : [CLS] the chef cooked the meal in the small kitchen and the guests enjoyed every bite of the delicious dinner [SEP]
마스킹    : [CLS] the [MASK] [MASK] the meal in the small [MASK] and the guests [MASK] every bite of the [MASK] dinner [SEP]
생성기 결과: [CLS] the guests eat the meal in the small box and the guests enjoy every bite of the whole dinner [SEP]


### 6-2. RTD 정답 라벨 만들기 + 판별기로 탐지하기

- 정답 라벨: `corrupted_ids[i] != orig_ids[i]` 이면 1(교체됨), 같으면 0(원본)
- 판별기 출력: `ElectraForPreTraining` 의 logits 은 토큰별 스칼라 값이며,
  **0보다 크면 '교체됨'** 으로 해석합니다.

In [62]:
rtd_labels = (corrupted_ids != orig_ids).long()

with torch.no_grad():
    disc_logits = discriminator(input_ids=corrupted_ids).logits    # (1, L)

rtd_pred = (disc_logits > 0).long()

toks = e_tok.convert_ids_to_tokens(corrupted_ids[0])
print(f"{'idx':>4} {'토큰':>12} {'정답':>6} {'예측':>6} {'logit':>8}")
for i, t in enumerate(toks):
    mark = "  <-- 교체" if int(rtd_labels[0, i]) == 1 else ""
    print(f"{i:>4} {t:>12} {int(rtd_labels[0,i]):>6} {int(rtd_pred[0,i]):>6} "
          f"{float(disc_logits[0,i]):>8.2f}{mark}")

acc = float((rtd_pred == rtd_labels).float().mean())
print(f"\n판별기 토큰 단위 정확도: {acc:.2%}")

 idx           토큰     정답     예측    logit
   0        [CLS]      0      0   -10.74
   1          the      0      0    -4.28
   2       guests      1      0    -3.27  <-- 교체
   3          eat      1      0    -2.86  <-- 교체
   4          the      0      0    -1.92
   5         meal      0      0    -2.18
   6           in      0      0    -3.70
   7          the      0      0    -1.49
   8        small      0      0    -1.78
   9          box      1      0    -2.25  <-- 교체
  10          and      0      0    -2.56
  11          the      0      0    -4.63
  12       guests      0      0    -3.33
  13        enjoy      1      0    -3.06  <-- 교체
  14        every      0      0    -3.92
  15         bite      0      0    -3.35
  16           of      0      0    -2.93
  17          the      0      0    -4.63
  18        whole      1      0    -1.73  <-- 교체
  19       dinner      0      0    -1.26
  20        [SEP]      0      0   -10.75

판별기 토큰 단위 정확도: 76.19%


### 6-3. MLM과 RTD의 학습 신호 비교

BERT의 MLM은 전체 토큰의 15%에서만 손실이 계산되지만,
ELECTRA의 RTD는 **모든 토큰**에서 손실이 계산됩니다. 그 차이를 숫자로 확인하세요.

In [63]:
L_ = orig_ids.shape[1]

n_signal_mlm = L_ * 0.15      # BERT MLM (전체의 15%) — 실수로 두세요
n_signal_rtd = L_      # ELECTRA RTD (전체 토큰)

print(f"문장 길이            : {L_}")
print(f"MLM 학습 신호 위치   : {n_signal_mlm:.1f}")
print(f"RTD 학습 신호 위치   : {n_signal_rtd}")
print(f"신호량 비율          : {n_signal_rtd / n_signal_mlm:.1f}배")

문장 길이            : 21
MLM 학습 신호 위치   : 3.1
RTD 학습 신호 위치   : 21
신호량 비율          : 6.7배


In [64]:
# ===== 자가 채점 (문제 6) =====
assert corrupted_ids.shape == orig_ids.shape
assert int((corrupted_ids == e_tok.mask_token_id).sum()) == 0, "[MASK]가 남아 있으면 안 됩니다."
unchanged = [i for i in range(orig_ids.shape[1]) if i not in MASK_POS]
assert torch.equal(corrupted_ids[0, unchanged], orig_ids[0, unchanged]), \
    "마스킹하지 않은 위치는 바뀌면 안 됩니다."

assert rtd_labels.shape == orig_ids.shape and set(rtd_labels.flatten().tolist()) <= {0, 1}
assert torch.equal(rtd_labels, (corrupted_ids != orig_ids).long()), \
    "생성기 예측이 원본과 같으면 라벨은 0(원본)이어야 합니다."
assert int(rtd_labels.sum()) <= len(MASK_POS), \
    "교체된 토큰 수가 마스킹한 개수를 넘을 수 없습니다."
assert torch.equal(rtd_pred, (disc_logits > 0).long()), "판별기 임계값은 0 입니다."
assert acc >= 0.6, f"판별기 정확도가 너무 낮습니다: {acc:.2%}"

assert n_signal_rtd == L_, "RTD 는 모든 토큰에서 손실이 계산됩니다."
assert abs(n_signal_rtd / n_signal_mlm - 1 / 0.15) < 0.5, \
    "MLM 은 전체의 15% 에서만 손실이 계산됩니다 (약 6.7배 차이)."
print("문제 6: 모든 검증 통과")

문제 6: 모든 검증 통과


---
# 문제 7. T5 — 모든 과업을 Text-to-Text로

T5는 지금까지의 사전 학습 언어 모델을 총체적으로 분석하여,
**자연어 처리 과업을 텍스트를 입력받아 텍스트를 출력하는 문제로 통일**했습니다.
입력 앞에는 CTRL과 같이 **제어(접두) 코드**가 붙습니다.

| 과업 | 입력 | 출력 |
|------|------|------|
| 번역 | `translate English to German: That is good` | `Das ist gut` |
| 문법성 판단(CoLA) | `cola sentence: The course is jumping well.` | `not acceptable` (체크포인트에 따라 `unacceptable`) |
| 유사도(STS-B) | `stsb sentence1: ... sentence2: ...` | `3.8` |

### 7-1. 접두 코드를 붙여 하나의 모델로 세 과업을 수행하세요.

In [65]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("t5-small").eval()

def t5_run(text, max_new_tokens=32, keep_special=False):
    enc = t5_tok(text, return_tensors="pt")
    with torch.no_grad():
        out = t5.generate(**enc, max_new_tokens=max_new_tokens)
    return t5_tok.decode(out[0], skip_special_tokens=not keep_special)


task_translate = "translate English to German: That is good"    # "That is good" 를 독일어로 번역
task_cola      = "cola sentence: The course is jumping well."    # "The course is jumping well." 의 문법성 판단
task_stsb      = "stsb sentence1: The rhino grazed on the grass. sentence2: A rhino is grazing in a field."    # 두 문장의 유사도
#   sentence1: The rhino grazed on the grass.
#   sentence2: A rhino is grazing in a field.

for name, inp in [("번역", task_translate), ("CoLA", task_cola), ("STS-B", task_stsb)]:
    print(f"[{name}]\n  입력: {inp}\n  출력: {t5_run(inp)}\n")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1415.14it/s]


[번역]
  입력: translate English to German: That is good
  출력: Das ist gut.

[CoLA]
  입력: cola sentence: The course is jumping well.
  출력: acceptable

[STS-B]
  입력: stsb sentence1: The rhino grazed on the grass. sentence2: A rhino is grazing in a field.
  출력: 4.0



### 7-2. 다양한 길이의 빈칸 채우기 (Span Corruption)

T5는 `<extra_id_0>`, `<extra_id_1>` 같은 **센티넬 토큰**으로 표시된 빈칸을 채우도록 사전 학습되었습니다.
BERT의 `[MASK]`가 정확히 토큰 1개에 대응하는 것과 달리, T5의 빈칸 하나는 **여러 개의 토큰**으로 채워질 수 있습니다.

In [66]:
#       원문: "I love peanut butter and jelly sandwiches."
span_input = "I love peanut butter and <extra_id_0> sandwiches."

print("입력 :", span_input)
print("출력 :", t5_run(span_input, keep_special=True))   # 센티넬 토큰까지 보이도록
print()

# 빈칸이 두 개인 경우
two_blanks = "The <extra_id_0> is the capital of <extra_id_1>."
print("입력 :", two_blanks)
print("출력 :", t5_tok.decode(
    t5.generate(**t5_tok(two_blanks, return_tensors="pt"), max_new_tokens=32)[0],
    skip_special_tokens=False))

입력 : I love peanut butter and <extra_id_0> sandwiches.
출력 : <pad> I love peanut butter and peanut butter and peanut butter and peanut butter and peanut butter sandwich. I love peanut butter and peanut butter and peanut butter and peanut butter and

입력 : The <extra_id_0> is the capital of <extra_id_1>.
출력 : <pad> The capital of The Capital of The Capital of The Capital of the.</s>


In [67]:
# ===== 자가 채점 (문제 7) =====
assert task_translate.startswith("translate English to German:"), "번역 접두 코드를 확인하세요."
assert "That is good" in task_translate
assert task_cola.startswith("cola sentence:"), "CoLA 접두 코드를 확인하세요."
assert task_stsb.startswith("stsb sentence1:") and "sentence2:" in task_stsb, \
    "STS-B 입력은 'stsb sentence1: ... sentence2: ...' 형식입니다."

out_tr = t5_run(task_translate).lower()
assert "gut" in out_tr, f"독일어 번역 결과가 이상합니다: {out_tr}"
out_cola = t5_run(task_cola).strip().lower()
assert "acceptable" in out_cola, f"CoLA 출력: {out_cola}"   # acceptable / not acceptable / unacceptable
out_sts = t5_run(task_stsb).strip()
assert any(c.isdigit() for c in out_sts), f"STS-B 출력은 점수여야 합니다: {out_sts}"

assert "<extra_id_0>" in span_input and "jelly" not in span_input, \
    "'jelly' 자리를 <extra_id_0> 로 대체해야 합니다."
print("번역:", out_tr, "| CoLA:", out_cola, "| STS-B:", out_sts)
print("문제 7: 모든 검증 통과")

번역: das ist gut. | CoLA: acceptable | STS-B: 4.0
문제 7: 모든 검증 통과


---
# 문제 8. 사전 학습 → 하위 과업 미세조정 (감성 분석)

지금까지는 **사전 학습된** 모델을 그대로 사용했습니다.
이제 강의자료의 "GPT의 하위 과업 미세조정"에서처럼, 사전 학습 모델 위에 분류용 선형 변환을 얹고
실제 데이터로 **미세조정(Fine-Tuning)** 합니다.

- 데이터: **SST-2** (영화 리뷰 감성 분석, 강의자료의 "하나의 문장 분류 과업")
- 모델: **DistilBERT** (BERT보다 가볍고 빨라 로컬 GPU 실습에 적합)

> VS Code에서 `rl2026` 커널을 선택하세요. RTX 4070 Ti SUPER 기준 약 15초 정도 걸릴 수 있습니다.

In [68]:
from datasets import load_dataset

try:
    ds = load_dataset("stanfordnlp/sst2")
except Exception:
    ds = load_dataset("glue", "sst2")

N_TRAIN, N_EVAL = 4000, 500
train_raw = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
eval_raw  = ds["validation"].select(range(N_EVAL))

print(ds)
for i in range(3):
    print(f"  label={train_raw[i]['label']}  |  {train_raw[i]['sentence'].strip()}")

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
  label=1  |  klein , charming in comedies like american pie and dead-on in election ,
  label=1  |  be fruitful
  label=1  |  soulful and


In [69]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
ft_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

def encode(batch):
    return ft_tok(batch["sentence"], truncation=True, max_length=64,
                  padding="max_length")

cols = ["input_ids", "attention_mask", "label"]
train_ds = train_raw.map(encode, batched=True).select_columns(cols).with_format("torch")
eval_ds  = eval_raw.map(encode,  batched=True).select_columns(cols).with_format("torch")

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
eval_dl  = DataLoader(eval_ds,  batch_size=64)
print("batches:", len(train_dl), len(eval_dl))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3201.59it/s]


batches: 125 8


### 8-1. 평가 함수를 완성하세요.

In [70]:
@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    correct = total = 0
    for batch in dl:
        labels = batch["label"].to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "label"}
        logits = model(**inputs).logits            # (B, 2)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum())
        total += labels.numel()
    return correct / max(total, 1)


acc_before = evaluate(model, eval_dl)
print(f"미세조정 전 정확도: {acc_before:.2%}   (무작위 초기화된 분류 헤드 -> 50% 근처)")

미세조정 전 정확도: 47.00%   (무작위 초기화된 분류 헤드 -> 50% 근처)


### 8-2. 학습 루프를 완성하세요.

PyTorch 학습 루프의 4단계를 기억하세요:
`옵티마이저 초기화 → 순전파(손실 계산) → 역전파 → 파라미터 갱신`

In [71]:
import gc
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS, LR = 1, 5e-5
optimizer = AdamW(model.parameters(), lr=LR)
num_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, 0, num_steps)

model.train()
step = 0
t0 = time.perf_counter()
for epoch in range(EPOCHS):
    for batch in train_dl:
        labels = batch["label"].to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "label"}

        optimizer.zero_grad()

        loss = model(**inputs, labels=labels).loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        step += 1
        if step % 25 == 0:
            print(f"  step {step:4d}/{num_steps}  loss={loss.item():.4f}")

print(f"학습 완료 ({time.perf_counter() - t0:.1f}초)")

  step   25/125  loss=0.5265
  step   50/125  loss=0.4487
  step   75/125  loss=0.2908
  step  100/125  loss=0.4960
  step  125/125  loss=0.2867
학습 완료 (4.5초)


In [72]:
acc_after = evaluate(model, eval_dl)
print(f"미세조정 전 : {acc_before:.2%}")
print(f"미세조정 후 : {acc_after:.2%}   (+{(acc_after - acc_before) * 100:.1f}%p)")

# 직접 문장을 넣어 확인
model.eval()
samples = ["what a waste of two hours of my life .",
           "a beautifully crafted film with warmth and wit .",
           "the plot is predictable but the acting saves it ."]
enc = ft_tok(samples, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1)
for s, p in zip(samples, probs):
    label = "긍정" if p[1] > p[0] else "부정"
    print(f"  [{label} {max(p).item():.2f}] {s}")

미세조정 전 : 47.00%
미세조정 후 : 86.00%   (+39.0%p)
  [부정 0.95] what a waste of two hours of my life .
  [긍정 0.99] a beautifully crafted film with warmth and wit .
  [부정 0.87] the plot is predictable but the acting saves it .


In [73]:
# ===== 자가 채점 (문제 8) =====
assert 0.35 < acc_before < 0.70, "미세조정 전에는 무작위 수준(50% 근처)이어야 합니다."
assert acc_after > 0.80, f"미세조정 후 정확도가 너무 낮습니다: {acc_after:.2%}"
assert acc_after - acc_before > 0.20, "미세조정으로 성능이 크게 올라야 합니다."
print("문제 8: 모든 검증 통과")

문제 8: 모든 검증 통과


---
## 정리

| 관점 | 핵심 정리 |
|------|-----------|
| 구조 | 인코더(이해) / 디코더(생성) / 인코더-디코더(seq2seq) |
| 학습 목표 | MLM(BERT) → 동적 마스킹(RoBERTa) → Span 마스킹(SpanBERT) → RTD(ELECTRA) |
| 양적 확대 | GPT 1.17억 → GPT-2 15.42억 모수, BERT 16GB → RoBERTa 160GB 데이터 |
| 경량화 | 임베딩 분해·모수 공유(ALBERT), 지식 증류(DistilBERT), 병목 구조(MobileBERT) |
| 통합 | 모든 과업을 텍스트-투-텍스트로(T5), 노이즈 복원(BART), 제어 코드(CTRL) |
| 생태계 | HuggingFace `transformers` 로 위 모델들을 동일한 API에서 사용 |

## 심화 과제 (선택)

1. 문제 3의 `mlm_mask` 를 **SpanBERT 방식**으로 바꿔 보세요.
   (기하 분포로 span 길이를 뽑아 연속된 토큰을 통째로 마스킹)
2. 문제 8을 `bert-base-uncased`, `google/electra-small-discriminator`,
   `albert-base-v2` 로 각각 반복하여 **정확도 / 학습 시간 / 파라미터 수**를 표로 비교하세요.
3. 한국어 모델(`klue/bert-base`)과 NSMC 데이터로 문제 8을 다시 수행해 보세요.
4. BART(`facebook/bart-base`)로 문서 요약을, MarianMT로 번역을 수행하고
   T5의 결과와 비교해 보세요.

## 심화 과제 1. SpanBERT 방식 연속 구간 마스킹

기하분포로 span 길이를 샘플링하고, 선택한 연속 토큰을 예측 대상으로 지정합니다.

In [75]:
def span_mlm_mask(input_ids, tokenizer, mlm_prob=0.15, mean_span_length=3.0, rng=None):
    if rng is None:
        rng = torch.Generator().manual_seed(0)
    masked = input_ids.clone()
    labels = torch.full_like(input_ids, -100)
    special = torch.tensor(tokenizer.get_special_tokens_mask(
        input_ids.tolist(), already_has_special_tokens=True), dtype=torch.bool)
    target_count = max(1, int((~special).sum().item() * mlm_prob))
    selected = torch.zeros_like(input_ids, dtype=torch.bool)
    attempts = 0
    while int(selected.sum()) < target_count and attempts < input_ids.numel() * 4:
        start = int(torch.randint(input_ids.numel(), (1,), generator=rng))
        if special[start] or selected[start]:
            attempts += 1
            continue
        length = max(1, int(torch.distributions.Geometric(
            torch.tensor(1.0 / mean_span_length)).sample().item()))
        end = min(input_ids.numel(), start + length)
        selected[start:end] |= ~special[start:end]
        attempts += 1
    selected &= ~special
    labels[selected] = input_ids[selected]
    mask_choice = torch.bernoulli(torch.full_like(input_ids, 0.8, dtype=torch.float32), generator=rng).bool() & selected
    random_choice = torch.bernoulli(torch.full_like(input_ids, 0.5, dtype=torch.float32), generator=rng).bool() & selected & ~mask_choice
    masked[mask_choice] = tokenizer.mask_token_id
    random_tokens = torch.randint(len(tokenizer), input_ids.shape, dtype=torch.long, generator=rng)
    masked[random_choice] = random_tokens[random_choice]
    return masked, labels

span_ids = tok("the quick brown fox jumps over the lazy dog", return_tensors="pt")["input_ids"][0]
span_ids_masked, span_labels = span_mlm_mask(span_ids, tok, rng=torch.Generator().manual_seed(SEED))
print("원문  :", tok.decode(span_ids))
print("Span :", tok.decode(span_ids_masked))
print("예측 위치: ", (span_labels != -100).nonzero().flatten().tolist())


원문  : [CLS] the quick brown fox jumps over the lazy dog [SEP]
Span : [CLS] the quick brown fox jumps [MASK] the lazy dog [SEP]
예측 위치:  [6]


## 심화 과제 2. 사전학습 모델 비교

BERT, ELECTRA, ALBERT를 동일한 SST-2 일부 데이터로 비교합니다. 각 모델의 파라미터 수, 학습 시간, 평가 정확도를 기록합니다.

In [78]:
import gc
from torch.optim import AdamW

# 앞선 문제에서 사용한 모델을 정리해 Windows 페이징 파일 부족을 방지합니다.
for _name in ["bert", "mlm_model", "gpt2", "alb_m", "dist_m", "generator",
              "discriminator", "t5", "model"]:
    globals().pop(_name, None)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def timed_finetune(model_name, epochs=1, max_train=1000, max_eval=500, batch_size=8):
    train_part = train_raw.select(range(min(max_train, len(train_raw))))
    eval_part = eval_raw.select(range(min(max_eval, len(eval_raw))))
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    def enc(batch):
        return tokenizer(batch["sentence"], truncation=True, max_length=64, padding="max_length")
    train_set = train_part.map(enc, batched=True).select_columns(["input_ids", "attention_mask", "label"]).with_format("torch")
    eval_set = eval_part.map(enc, batched=True).select_columns(["input_ids", "attention_mask", "label"]).with_format("torch")
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(eval_set, batch_size=batch_size * 2)
    optimizer = AdamW(model.parameters(), lr=5e-5)
    try:
        start = time.perf_counter()
        model.train()
        for _ in range(epochs):
            for batch in train_loader:
                labels = batch.pop("label").to(DEVICE)
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                optimizer.zero_grad(set_to_none=True)
                loss = model(**inputs, labels=labels).loss
                loss.backward()
                optimizer.step()
        if DEVICE == "cuda": torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        accuracy = evaluate(model, eval_loader)
        return {"model": model_name, "parameters": sum(p.numel() for p in model.parameters()),
                "seconds": round(seconds, 2), "accuracy": round(accuracy, 4)}
    finally:
        del model, optimizer, train_loader, eval_loader, tokenizer
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()

comparison_models = ["bert-base-uncased", "google/electra-small-discriminator", "albert-base-v2"]
comparison_results = [timed_finetune(name) for name in comparison_models]
for row in comparison_results:
    print(f"{row['model']:<40} params={row['parameters']:,} time={row['seconds']:.2f}s acc={row['accuracy']:.2%}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1248.51it/s]
c:\Users\neton\anaconda3\envs\rl2026\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\neton\.cache\huggingface\hub\models--google--electra-small-discriminator. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 500/50

bert-base-uncased                        params=109,483,778 time=4.42s acc=64.40%
google/electra-small-discriminator       params=13,549,314 time=1.84s acc=57.80%
albert-base-v2                           params=11,685,122 time=2.99s acc=56.00%


## 심화 과제 3. 한국어 

`klue/bert-base`와 NSMC 데이터셋을 사용합니다. 데이터셋 다운로드가 필요합니다.

In [80]:
import io
import urllib.request
from datasets import Dataset, DatasetDict


def load_nsmc_split(url):
    with urllib.request.urlopen(url) as response:
        text = response.read().decode("utf-8")
    rows = []
    for line in io.StringIO(text).read().splitlines()[1:]:
        _, document, label = line.split("\t", 2)
        if document.strip():
            rows.append({"document": document, "label": int(label)})
    return Dataset.from_list(rows)


nsmc = DatasetDict({
    "train": load_nsmc_split("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"),
    "test": load_nsmc_split("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"),
})
nsmc_train = nsmc["train"].shuffle(seed=SEED).select(range(min(3000, len(nsmc["train"]))))
nsmc_test = nsmc["test"].select(range(min(500, len(nsmc["test"]))))
ko_tok = AutoTokenizer.from_pretrained("klue/bert-base")
ko_model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2).to(DEVICE)
def encode_ko(batch):
    return ko_tok(batch["document"], truncation=True, max_length=128, padding="max_length")
ko_train = nsmc_train.map(encode_ko, batched=True).rename_column("label", "labels").select_columns(["input_ids", "attention_mask", "labels"]).with_format("torch")
ko_test = nsmc_test.map(encode_ko, batched=True).rename_column("label", "labels").select_columns(["input_ids", "attention_mask", "labels"]).with_format("torch")
ko_train_loader = DataLoader(ko_train, batch_size=32, shuffle=True)
ko_test_loader = DataLoader(ko_test, batch_size=64)
ko_optimizer = AdamW(ko_model.parameters(), lr=5e-5)
ko_model.train()
for batch in ko_train_loader:
    labels = batch.pop("labels").to(DEVICE)
    inputs = {k: v.to(DEVICE) for k, v in batch.items()}
    ko_optimizer.zero_grad(set_to_none=True)
    loss = ko_model(**inputs, labels=labels).loss
    loss.backward()
    ko_optimizer.step()
ko_model.eval(); correct = total = 0
with torch.no_grad():
    for batch in ko_test_loader:
        labels = batch.pop("labels").to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        correct += int((ko_model(**inputs).logits.argmax(-1) == labels).sum())
        total += labels.numel()
print(f"NSMC test accuracy: {correct / total:.2%}")


c:\Users\neton\anaconda3\envs\rl2026\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\neton\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\neton\anaconda3\envs\rl2026\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingfac

NSMC test accuracy: 84.60%


## 심화 과제 4. BART 요약과 MarianMT 번역

BART 계열 요약 모델과 MarianMT 번역 모델을 실행해 T5 결과와 비교합니다. 모델 용량이 크므로 처음에는 다운로드 시간이 걸릴 수 있습니다.

In [2]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

summary_text = ("The transformer architecture changed natural language processing by using attention. "
                "Encoder models such as BERT understand context, while decoder models such as GPT generate text.")


def clear_runtime_models():
    for name in ["ko_model", "model", "bart", "mt", "gpt2", "mlm_model", "bert"]:
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


clear_runtime_models()
bart_name = "sshleifer/distilbart-cnn-12-6"
bart_tok = AutoTokenizer.from_pretrained(bart_name)
bart = AutoModelForSeq2SeqLM.from_pretrained(bart_name).to(DEVICE).eval()
bart_inputs = bart_tok(summary_text, return_tensors="pt", truncation=True).to(DEVICE)
with torch.inference_mode():
    bart_ids = bart.generate(**bart_inputs, max_new_tokens=40, num_beams=2)
bart_summary = bart_tok.decode(bart_ids[0], skip_special_tokens=True)
print("BART 요약:", bart_summary)
del bart_inputs, bart_ids, bart, bart_tok
clear_runtime_models()

mt_name = "Helsinki-NLP/opus-mt-en-de"
mt_tok = AutoTokenizer.from_pretrained(mt_name)
mt = AutoModelForSeq2SeqLM.from_pretrained(mt_name).to(DEVICE).eval()
mt_inputs = mt_tok("That is good.", return_tensors="pt").to(DEVICE)
with torch.inference_mode():
    mt_ids = mt.generate(**mt_inputs, max_new_tokens=32, num_beams=2)
print("MarianMT 번역:", mt_tok.decode(mt_ids[0], skip_special_tokens=True))
del mt_inputs, mt_ids, mt, mt_tok
clear_runtime_models()


device: cuda


W1110 02:07:10.027000 30436 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\neton\anaconda3\envs\rl2026\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\neton\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development


BART 요약:  The transformer architecture changed natural language processing by using attention . Encoder models such as BERT understand context, while decoder models like GPT generate text . The transformational architecture was created by


c:\Users\neton\anaconda3\envs\rl2026\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\neton\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-de. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\neton\anaconda3\envs\rl2026\Lib\site-packages\transformers\models\marian\tokenization_marian.py:

MarianMT 번역: Das ist gut.
